In [1]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS

In [ ]:
# This notebook is an attempt to track down the source 
# of the electrical noise which causes the horizontal banding in AuxTel images.

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.14 sec
Read 1 history items for RemoteEvent(ATHeaderService, 0, authList)
Read 100 history items for RemoteEvent(ATHeaderService, 0, heartbeat)
Read 5 history items for RemoteEvent(ATHeaderService, 0, largeFileObjectAvailable)
Read 1 history items for RemoteEvent(ATHeaderService, 0, logLevel)
Read 100 history items for RemoteEvent(ATHeaderService, 0, logMessage)
Read 1 history items for RemoteEvent(ATHeaderService, 0, simulationMode)
Read 1 history items for RemoteEvent(ATHeaderService, 0, softwareVersions)
Read 5 history items for RemoteEvent(ATHeaderService, 0, summaryState)
Read historical data 

[[None, None, None, None, None, None, None], [None, None, None, None]]

positionStatus DDS read queue is filling: 12 of 100 elements
position DDS read queue is filling: 48 of 100 elements
timeAndDate DDS read queue is filling: 87 of 100 elements
m2AirPressure DDS read queue is filling: 13 of 100 elements
trajectory DDS read queue is filling: 23 of 100 elements
mount_positions DDS read queue is filling: 18 of 100 elements
m1AirPressure DDS read queue is filling: 13 of 100 elements
torqueDemand DDS read queue is filling: 23 of 100 elements
mountStatus DDS read queue is filling: 88 of 100 elements
loadCell DDS read queue is filling: 13 of 100 elements
guidingAndOffsets DDS read queue is filling: 88 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 23 of 100 elements
currentTargetStatus DDS read queue is filling: 89 of 100 elements
mount_Nasmyth_Encoders DDS read queue is filling: 23 of 100 elements
mount_AzEl_Encoders DDS read queue is filling: 23 of 100 elements
mount_AzEl_Encoders python read queue is filling: 22 of 100 elements
measu

In [5]:
# enable components
await atcs.enable()
await latiss.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdome': 'test', 'atdometrajectory': ''}
[atmcs]::[<State.ENABLED: 2>]
[atptg]::[<State.ENABLED: 2>]
[ataos]::[<State.ENABLED: 2>]
[atpneumatics]::[<State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::[<State.ENABLED: 2>]
[atdometrajectory]::[<State.ENABLED: 2>]
All components in <State.ENABLED: 

In [ ]:
# All components now enabled. - 16-Feb-2021 4:48 PM

In [6]:
# Ensure we're using Nasmyth 2
await atcs.rem.atptg.cmd_focusName.set_start(focus=3)

In [7]:
# Disable the dome
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.DISABLED, settingsToApply="test")
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.DISABLED)

[<State.ENABLED: 2>, <State.DISABLED: 1>]

In [8]:
# Take event checking out the slew commands to test telescope only
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [9]:
# Run 1 bias as a test
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 17 of 100 elements


array([2021021600001])

In [10]:
# Move telescope to park position (az=0, el=80, rot=0)
current_az = 0.0
current_el = 80.0
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +064.998 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +062.810 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +058.812 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +052.812 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +048.813 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +044.811 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +038.812

In [11]:
# Move telescope to (az=180, el=80, rot=0). More convenient positioning for photos
current_az = 180.0
current_el = 80.0
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got False
Telescope not in position
[Telescope] delta Alt = -000.000 deg; delta Az= +180.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +178.474 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +172.513 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +000.000 deg; delta Az= +168.513 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +164.512 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +158.514 deg; delta N1 = -000.000 deg; delta N2 = +000.000 d

In [12]:
# Move the rot to one end of the travel
current_rot = 150.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +149.844 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +144.784 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +138.786 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +132.785 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +128.784 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +122.789 deg 
[Telescope] delta Alt = -000.000

In [13]:
# Move the rot to other end of the travel
current_rot = -150.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +060.017 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +064.436 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +068.438 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +074.436 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +078.438 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +084.437 deg 
[Telescope] delta Alt = -000.000

In [14]:
# Move the rot to other end of the travel
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +149.911 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +145.061 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +139.064 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +135.059 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +129.065 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +123.063 deg 
[Telescope] delta Alt = -000.000

In [15]:
# Now run 50 biases, with 2 second delay. Images 2-51
for i in range(50):
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
    await asyncio.sleep(2)
    await latiss.take_bias(1)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
Axes in position.
Generating group_id
BIAS 0001 - 0001
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
Axes in position.
Generating group_id
BIAS 00

In [16]:
# Move the rot to +150
current_rot = 150.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +149.904 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +145.053 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +139.049 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +135.050 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +129.046 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +123.050 deg 
[Telescope] delta Alt = -000.000

In [17]:
# Now run 50 biases, with 2 sec delay between telescope rot changes
# but moving through the whole range.  There was some indication in the on-sky tests
# that negative rot angles were worse.
# Images 52-101
current_rot = 150.0
for i in range(50):
    final_delay = 2.0
    current_rot -= 6.0
    await atcs.point_azel(current_az, current_el, rot_tel=current_rot)
    await asyncio.sleep(final_delay)
    await latiss.take_bias(1)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -006.000 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -002.259 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.026 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
Telescope in position.
Generating group_id
BIAS 0001 - 0001
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...

In [18]:
# Move the rot to 0
current_rot = 0.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = +149.922 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +145.133 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +139.135 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +133.131 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = +127.131 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +121.133 deg 
[Telescope] delta Alt = -000.000

In [19]:
# Re-enable the dome
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [24]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply="test")

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [25]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [28]:
await atcs.prepare_for_flatfield()

Cover state <MirrorCoverState.OPENED: 7>
M1 cover already opened.
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.014 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = +000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -000.000 deg 
Telescope in position.
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.


RuntimeError: ATDome is deactivated. Activate it by setting `check.atdome=True` before slewing.In some cases users deactivate a component on purpose.Make sure it is clear to operate the dome before doing so.

In [29]:
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

Generating group_id
FLAT 0001 - 0010
FLAT 0002 - 0010
FLAT 0003 - 0010
FLAT 0004 - 0010
FLAT 0005 - 0010
FLAT 0006 - 0010
FLAT 0007 - 0010
logMessage DDS read queue is filling: 23 of 100 elements
FLAT 0008 - 0010
FLAT 0009 - 0010
FLAT 0010 - 0010


array([2021021600102, 2021021600103, 2021021600104, 2021021600105,
       2021021600106, 2021021600107, 2021021600108, 2021021600109,
       2021021600110, 2021021600111])

In [30]:
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 2, filter='empty_1', grating='empty_1')

Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
logMessage DDS read queue is filling: 11 of 100 elements
logMessage DDS read queue is filling: 11 of 100 elements
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
logMessage DDS read queue is filling: 13 of 100 elements
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id
FLAT 0001 - 0002
FLAT 0002 - 0002
Generating group_id

In [31]:
# Putting everything back in standby.
await atcs.shutdown()

Disabling ATAOS corrections
Disabling ATAOS corrections.
Closing M1 cover vent gates.
Cover state <MirrorCoverState.OPENED: 7>
Closing M1 cover.
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +030.985 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +030.153 deg; delta Az= -000.000 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +027.038 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +023.941 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.000 deg 
[Telescope] delta Alt = +018.362 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = +000.000 deg 
[Telescope] delta Alt = +012.476 deg; delta Az= +000.000 deg; delta N1 

In [32]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [33]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=2017570163, ack=<SalRetCode.CMD_FAILED: -302>, error=1, result='Failed: start not allowed in state <State.ENABLED: 2>')

In [34]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [35]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.STANDBY)

[<State.STANDBY: 5>]

In [36]:
await salobj.set_summary_state(latiss.rem.atspectrograph, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atcamera, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atheaderservice, salobj.State.STANDBY)
await salobj.set_summary_state(latiss.rem.atarchiver, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]